##  Load the open ai for agent 

- Give the tools as the list of dict。 The dict is the tool dict with the name description 。。。 

## with the action schema and the question ， LLM by understand the action schema description to get the list 

- Understand the output from the array to get the planer like such 
    response.output = [
    ResponseFunctionToolCall(
        type="function_call",
        name="get_weather",
        arguments='{"city":"北京","day":"tomorrow"}',
        call_id="call_1"
    ),
    ResponseFunctionToolCall(
        type="function_call",
        name="get_weather",
        arguments='{"city":"上海","day":"tomorrow"}',
        call_id="call_2"
    ),
    ResponseFunctionToolCall(
        type="function_call",
        name="calculator",
        arguments='{"expression":"25*17"}',
        call_id="call_3"
    )
]

- Based on this to call the function tools ， get the output list 

- output list will be the input to the llm，to get the answer （ Here should input the “question” i think ）  

In [ ]:

from openai import OpenAI
import json

client = OpenAI()

def calculator(expression: str) -> str:
    return str(eval(expression, {"__builtins__": {}}, {}))

def get_exchange_rate(base: str, target: str) -> str:
    # mock
    rates = {
        ("USD", "CNY"): 7.2,
        ("USD", "EUR"): 0.92
    }
    rate = rates.get((base, target), None)
    if rate is None:
        return "Rate not found"
    return str(rate)

tools = [
    {
        "type": "function",
        "name": "calculator",
        "description": "Evaluate a math expression",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get FX rate from one currency to another",
        "parameters": {
            "type": "object",
            "properties": {
                "base": {"type": "string"},
                "target": {"type": "string"}
            },
            "required": ["base", "target"],
            "additionalProperties": False
        },
        "strict": True
    }
]



user_input = "If 100 USD is converted to CNY, how much is it? Use exchange rate first, then calculator."


In [ ]:
# create contains the action schema, the model and the input query 
response = client.responses.create(
    
        model = 'gpt-5', 
        input = user_input , 
        tools = tools # list of the action tool 
)

# rsponse will need to parse the content 
# response.output is formulate  in the form of the api style 

tool_outputs = [ ]

for item in response.output: 
    # type, name, discrezation, 
     
    if item.type == 'function_call': 
        args = json.loads(item.arguments) 
        
        if item.name == 'get_exchange_rate': 
             
            out = get_exchange_rate(**args)
        elif item.name == "calculator":
            out = calculator(**args)
            
        else: 
            out = 'Unknow tools'
            
    # call it        
    tool_outputs.append({
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": out
    })
 
 

## 然后得到 结果 after call the function and orginize into a list 
# the LLM will take the output as the input to LLM 
# to output the response    
if tool_outputs:
    response2 = client.responses.create(
        model="gpt-5",
        previous_response_id=response.id,
        input=tool_outputs
    )
    print(response2.output_text)  
    
    

In [ ]:

## 然后得到 结果 after call the function and orginize into a list 
# the LLM will take the output as the input to LLM 
# to output the response    
if tool_outputs:
    response2 = client.responses.create(
        model="gpt-5",
        previous_response_id=response.id,
        input=tool_outputs
    )
    print(response2.output_text)  
    
    

## Not API call ， load the local model

-  write the message in the chatbot formulate

    exp： the message have the role and content for user and assistant paired data 
    messages = [
        {"role": "system", "content": "You are a helpful assistant that uses tools when needed."},
        {"role": "user", "content": "What is 25*17? Use the calculator tool."}
    ] 

    - apply_chat_template is very important in HuggingFace / tokenizer + chat model + tool calling 
    把结构化的 messages + tools → 转换成模型真正吃的 prompt 字符串. with the tokenizer.apply_chat_template, the message can be formulated into the text message. 

    exp: 
    1. <|system|>
        You are a helpful assistant that uses tools when needed.

        <|user|>
        What is 25*17? Use the calculator tool.

        <|assistant|>

    2. Tools apply . Add the action schema with the text form 

        <|system|>
        You are a helpful assistant that uses tools when needed.

        Available tools:
        calculator(expression)

        <|user|>
        What is 25*17? Use the calculator tool.

        <|assistant|> # this is for text generation 

    3. tokenizer -- model prediction -- decode to text --
         


    4. add the output to the message, which is the generated content after assistant and the called function output 
        **The reason it cant be like API to remember the output** The generated output will give use the tool. ... 
        - Additional result from calling the tool should also be the input to the message. Because this is language model it can't give the result directly?


        - 假设 parse_tool_call 能从 generated 里解析出 tool call
            tool_call = parse_tool_call(generated)

            if tool_call is None:
                print("FINAL ANSWER:")
                print(generated)
                break

            tool_name = tool_call["name"]
            args = tool_call["arguments"]

            result = TOOL_MAP[tool_name](**args)



In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-7B-Instruct"   # 示例
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

def calculator(expression: str) -> str:
    return str(eval(expression, {"__builtins__": {}}, {}))

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a math expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"],
                "additionalProperties": False
            }
        }
    }
]

messages = [
    {"role": "system", "content": "You are a helpful assistant that uses tools when needed."},
    {"role": "user", "content": "What is 25*17? Use the calculator tool."}
]

# tokenizer into the chat form “ apply_chat_template ” by the tokenizer 
text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("MODEL RAW OUTPUT:")
print(generated)


In [ ]:

# 下面这一步是关键：
# 你要根据所用模型的 tool-call 输出格式，自行解析 generated，
# 如果发现模型请求调用 calculator，就执行:

# have to give the reuslt 
result = calculator("25*17")

messages.append({
    "role": "assistant",
    "content": generated
})

messages.append({
    "role": "tool",
    "name": "calculator",
    "content": result
})



# 再来第二轮，让模型基于 tool output 产出最终自然语言回答
text2 = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)
inputs2 = tokenizer(text2, return_tensors="pt").to(model.device)
outputs2 = model.generate(**inputs2, max_new_tokens=128)
final_answer = tokenizer.decode(outputs2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)

print("FINAL ANSWER:")
print(final_answer) 